# 08_3_9 DQA Scene Phase2 R-SCoLQ Anti-Drift Policy

08_3_8 の評価が終わったので、ここではまず「なぜ性能が上がらないのか」を整理し、その仮説に基づいて次の実験を起動する。

08_3_8 の大きな収穫は、08_3_7 の `global_multiplier = 1.0 / 0.15` の振動を消せたこと。一方で、`r002` は良いが `r010` で落ちる問題は残った。

今回の仮説:

- raw R-SCoLQ は confidence よりは良いが、round が進むとモデル自身の pseudo box に対して raw R-SCoLQ / objectness が上がる。
- つまり `pseudo数が大きく増えない` かつ `平均R-SCoLQが上がる` 状態を、08_3_8 は良い状態として扱ってしまう。
- しかし評価では highway / citystreet / residential すべてで r010 が落ちるので、これは自己確認的な drift と見るべき。
- 08_3_9 は GT による best 保存ではなく、round が進むほど pseudo bbox の信頼を弱める anti-drift policy にする。

設計:

- anchor は 08_3_8 と同じく target round001 raw stats。
- pseudo budget と quality budget は「少し増えた時点」で罰則が入るように margin を負に寄せる。
- global/class multiplier の power を上げ、EMA を少し速くする。
- R-SCoLQ gate は high=0.70 にして、bbox positive をより厳しくする。
- pseudo bbox/cls loss と residual blend は round 後半で強く減衰させる。
- client は `neck_head` 更新にして、backbone drift を抑える。


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "dynamic_quality_aware_classwise_aggregation").exists() and (path / "navigating_data_heterogeneity").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase2_rscolq_anti_drift_policy.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
RSCOLQ_ROOT = DQA_ROOT / "source_calibrated_localization_quality"
RSCOLQ_MODEL = RSCOLQ_ROOT / "artifacts" / "rscolq_best.joblib"
PREV_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_8_phase2_rscolq_smooth_policy" / "a_rscolq_smooth_raw_ema_r003"
PREV_STATS_ROOT = DQA_ROOT / "stats_dqa08_3_8_phase2_rscolq_smooth_policy" / "a_rscolq_smooth_raw_ema_r003"
PREV_REPORT = PREV_WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"

BASE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_9_phase2_rscolq_anti_drift_policy"
BASE_STATS_ROOT = DQA_ROOT / "stats_dqa08_3_9_phase2_rscolq_anti_drift_policy"
BASE_LOG_ROOT = DQA_ROOT / "logs_dqa08_3_9_phase2_rscolq_anti_drift_policy"

for path in [RUN_SCRIPT, EVAL_SCRIPT, SOURCE_WORK_ROOT, RSCOLQ_MODEL, PREV_REPORT]:
    print(path, "exists=", path.exists())

for path in [BASE_WORK_ROOT, BASE_STATS_ROOT, BASE_LOG_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("workspace:", BASE_WORK_ROOT)
print("stats:", BASE_STATS_ROOT)
print("logs:", BASE_LOG_ROOT)


/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_rscolq_anti_drift_policy.py exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/artifacts/rscolq_best.joblib exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_8_phase2_rscolq_smooth_policy/a_rscolq_smooth_raw_ema_r003/validation_reports/paper_protocol_eval_summary.csv exists= True
workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_9_phase2_rscolq_anti_drift_policy
stats: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_9_phase2_rscolq_anti_drift_policy
logs: 

In [2]:
# 08_3_8 の確定結果を読む。
prev_eval = pd.read_csv(PREV_REPORT)
cols = ["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]

print("08_3_8 scene_total sorted")
display(prev_eval[prev_eval["split"] == "scene_total"][cols].sort_values(["map50_95", "map50"], ascending=False))

print("08_3_8 selected rounds by split")
sel = prev_eval[prev_eval["checkpoint_label"].str.contains("a_rscolq_smooth_raw_ema", regex=False)][cols]
display(sel.sort_values(["checkpoint_label", "split"]))

# 08_3_8 stats drift summary
classes = ["person", "rider", "car", "bus", "truck", "bike", "motor", "traffic light", "traffic sign", "train"]
rows = []
for round_idx in range(1, 11):
    path = PREV_STATS_ROOT / f"phase2_round{round_idx:03d}.json"
    if not path.exists():
        continue
    data = json.loads(path.read_text())
    clients = data.get("clients", [data])
    counts = [0.0] * 10
    q_sums = [0.0] * 10
    obj_sums = [0.0] * 10
    for client in clients:
        for i, value in enumerate(client.get("counts", [])):
            counts[i] += value
        for i, value in enumerate(client.get("quality_sums", [])):
            q_sums[i] += value
        for i, value in enumerate(client.get("objectness_sums", [])):
            obj_sums[i] += value
    total = sum(counts)
    rows.append({
        "round": round_idx,
        "pseudo_total": int(total),
        "mean_raw_rscolq": sum(q_sums) / total,
        "mean_objectness": sum(obj_sums) / total,
        "person": int(counts[0]),
        "car": int(counts[2]),
        "traffic_light": int(counts[7]),
        "traffic_sign": int(counts[8]),
        "active_classes": sum(c > 0 for c in counts),
    })

drift = pd.DataFrame(rows)
print("08_3_8 pseudo/stat drift")
display(drift)

out = BASE_WORK_ROOT / "analysis_0838_drift_summary.csv"
drift.to_csv(out, index=False)
print("saved:", out)


08_3_8 scene_total sorted


,checkpoint_label,split,precision,recall,map50,map50_95
15,old08_3_4_d_r002,scene_total,0.503,0.399,0.381,0.213
47,a_rscolq_smooth_raw_ema_r003_r002,scene_total,0.507,0.395,0.380,0.213
23,old08_3_6_a_r002,scene_total,0.489,0.411,0.381,0.212
31,old08_3_7_a_r001,scene_total,0.485,0.406,0.381,0.212
35,old08_3_7_a_r002,scene_total,0.494,0.406,0.381,0.212
11,old08_3_4_c_r002,scene_total,0.492,0.408,0.380,0.212
19,old08_3_6_a_r001,scene_total,0.493,0.401,0.382,0.211
43,a_rscolq_smooth_raw_ema_r003_r001,scene_total,0.489,0.404,0.381,0.211
3,p1_r003,scene_total,0.471,0.401,0.375,0.204
7,p1_r012,scene_total,0.743,0.326,0.368,0.199


08_3_8 selected rounds by split


,checkpoint_label,split,precision,recall,map50,map50_95
41,a_rscolq_smooth_raw_ema_r003_r001,citystreet,0.488,0.411,0.386,0.215
40,a_rscolq_smooth_raw_ema_r003_r001,highway,0.464,0.379,0.336,0.188
42,a_rscolq_smooth_raw_ema_r003_r001,residential,0.418,0.468,0.422,0.238
43,a_rscolq_smooth_raw_ema_r003_r001,scene_total,0.489,0.404,0.381,0.211
45,a_rscolq_smooth_raw_ema_r003_r002,citystreet,0.506,0.402,0.386,0.216
44,a_rscolq_smooth_raw_ema_r003_r002,highway,0.472,0.369,0.332,0.188
46,a_rscolq_smooth_raw_ema_r003_r002,residential,0.518,0.417,0.418,0.238
47,a_rscolq_smooth_raw_ema_r003_r002,scene_total,0.507,0.395,0.380,0.213
49,a_rscolq_smooth_raw_ema_r003_r010,citystreet,0.516,0.380,0.367,0.201
48,a_rscolq_smooth_raw_ema_r003_r010,highway,0.505,0.352,0.319,0.175


08_3_8 pseudo/stat drift


,round,pseudo_total,mean_raw_rscolq,mean_objectness,person,car,traffic_light,traffic_sign,active_classes
0,1,708002,0.649214,0.378617,87373,255214,109182,217244,8
1,2,707671,0.657599,0.390911,82664,255359,109748,218575,8
2,3,697317,0.649835,0.386040,84344,255426,106004,212448,8
3,4,697250,0.652921,0.388435,83023,255262,106578,213341,8
4,5,702804,0.655117,0.390201,84445,255298,107012,215603,8
5,6,710295,0.663563,0.396777,87129,255423,109322,218843,8
6,7,718700,0.667302,0.400016,87782,255471,115331,218863,9
7,8,720842,0.672075,0.405771,91132,255574,113772,219509,8
8,9,717172,0.676200,0.412273,92147,255723,111379,218217,10
9,10,719300,0.683610,0.423367,93472,255769,111066,218236,9


saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_9_phase2_rscolq_anti_drift_policy/analysis_0838_drift_summary.csv


## 08_3_9 Variant

`a_rscolq_antidrift_budget_r003` をまず1本だけ走らせる。

08_3_8 は `global_multiplier` が最後まで 1.0 のままだったので、今回は以下を変える。

- `GLOBAL_BUDGET_MARGIN=-0.01`: round001 の 99% を超えた時点から pseudo 数の罰則対象。
- `QUALITY_MARGIN=-0.005`: 平均 raw R-SCoLQ が anchor より少し上がるだけでも自己確認的 drift として扱う。
- `GLOBAL_POWER=2.0`, `QUALITY_POWER=4.0`, `EMA=0.50`: 罰則を強く、反応を速くする。
- `HIGH=0.70`, `MAX_PSEUDO_PER_IMAGE=55`, `MAX_PSEUDO_PER_CLASS_IMAGE=14`: bbox positive と pseudo 上限を絞る。
- `neck_head` client update, residual/min_server_alpha decay: backbone drift を抑える。


In [3]:
PHASE2_ROUNDS_PER_VARIANT = 10
BATCH_SIZE = 160
WORKERS = 10
GPUS = 2
MASTER_PORT_BASE = 30190
STREAM_TRAIN_OUTPUT = False
MIN_FREE_GIB = 30

SELECTED_VARIANTS = ["a_rscolq_antidrift_budget_r003"]

COMMON_ANTI_DRIFT_ENV = {
    "DQA0839_RSCOLQ_ANCHOR_ROUND": 1,
    "DQA0839_RSCOLQ_GLOBAL_BUDGET_MARGIN": -0.01,
    "DQA0839_RSCOLQ_QUALITY_MARGIN": -0.005,
    "DQA0839_RSCOLQ_GLOBAL_POWER": 2.0,
    "DQA0839_RSCOLQ_QUALITY_POWER": 4.0,
    "DQA0839_RSCOLQ_CLASS_BUDGET_MARGIN": -0.02,
    "DQA0839_RSCOLQ_CLASS_POWER": 0.45,
    "DQA0839_RSCOLQ_MIN_MULTIPLIER": 0.62,
    "DQA0839_RSCOLQ_MULTIPLIER_EMA": 0.50,
}

VARIANTS = [
    {
        "name": "a_rscolq_antidrift_budget_r003",
        "description": "08_3_9本命。raw R-SCoLQ self-confirmationを早めに罰し、neck/head更新とpseudo loss decayで長期driftを抑える。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "neck_head",
        "classwise_blend": 0.035,
        "server_anchor": 20.0,
        "temperature": 3.2,
        "stability_lambda": 0.78,
        "residual_start": 0.06,
        "residual_end": 0.01,
        "min_server_alpha_start": 0.80,
        "min_server_alpha_end": 0.90,
        "env": {
            **COMMON_ANTI_DRIFT_ENV,
            "DQA0839_RSCOLQ_LOW": 0.18,
            "DQA0839_RSCOLQ_MID": 0.42,
            "DQA0839_RSCOLQ_HIGH": 0.70,
            "DQA0839_NMS_CONF_THRES": 0.012,
            "DQA0839_TEACHER_LOSS_WEIGHT": 0.24,
            "DQA0839_BOX_LOSS_WEIGHT_START": 0.005,
            "DQA0839_BOX_LOSS_WEIGHT_END": 0.0008,
            "DQA0839_OBJ_LOSS_WEIGHT": 0.24,
            "DQA0839_CLS_LOSS_WEIGHT": 0.035,
            "DQA0839_LOW_MID_OBJ_WEIGHT": 0.10,
            "DQA0839_MID_HIGH_OBJ_WEIGHT": 0.42,
            "DQA0839_CLIENT_LR0_START": 0.00065,
            "DQA0839_CLIENT_LR0_END": 0.00008,
            "DQA0839_SERVER_LR0_START": 0.0020,
            "DQA0839_SERVER_LR0_END": 0.00055,
            "DQA0839_MAX_PSEUDO_PER_IMAGE": 55,
            "DQA0839_MAX_PSEUDO_PER_CLASS_IMAGE": 14,
        },
    },
]

selected = [v for v in VARIANTS if not SELECTED_VARIANTS or v["name"] in SELECTED_VARIANTS]
print("selected:", [v["name"] for v in selected])


selected: ['a_rscolq_antidrift_budget_r003']


In [4]:
def variant_work_root(variant: dict) -> Path:
    return BASE_WORK_ROOT / variant["name"]


def variant_stats_root(variant: dict) -> Path:
    return BASE_STATS_ROOT / variant["name"]


def variant_runner_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_runner.out"


def variant_train_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_train.log"


def variant_pid_path(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}.pid"


def variant_env(variant: dict) -> dict[str, str]:
    env = os.environ.copy()
    stats_root = variant_stats_root(variant)
    stats_root.mkdir(parents=True, exist_ok=True)
    env["DQA0839_VARIANT"] = variant["name"]
    env["DQA0839_SOURCE_WORK_ROOT"] = str(SOURCE_WORK_ROOT)
    env["DQA0839_SOURCE_PHASE1_ROUND"] = str(variant["source_phase1_round"])
    env["DQA0839_RSCOLQ_MODEL"] = str(RSCOLQ_MODEL)
    env["DQA0839_CLIENT_TRAIN_SCOPE"] = variant["client_train_scope"]
    env["DQA0839_SERVER_TRAIN_SCOPE"] = "all"
    env["DQA0839_RESIDUAL_START"] = str(variant["residual_start"])
    env["DQA0839_RESIDUAL_END"] = str(variant["residual_end"])
    env["DQA0839_MIN_SERVER_ALPHA_START"] = str(variant["min_server_alpha_start"])
    env["DQA0839_MIN_SERVER_ALPHA_END"] = str(variant["min_server_alpha_end"])
    env["DQA0839_PHASE2_ROUNDS"] = str(PHASE2_ROUNDS_PER_VARIANT)
    env["DQA08_STATS_ROOT"] = str(stats_root.resolve())
    env["DQA08_THRESHOLD_LOG"] = str((stats_root / "phase2_rscolq_anti_drift_policy_schedule.jsonl").resolve())
    env["DQA_STATS_QUALITY_MODE"] = "rscolq_raw"
    env["DQA0834_STATS_QUALITY_MODE"] = "rscolq_raw"
    env.setdefault("ET_SKIP_AFTER_TRAIN_BEST_VAL", "1")
    for key, value in variant.get("env", {}).items():
        env[key] = str(value)
    return env


def train_cmd(variant: dict) -> list[str]:
    return [
        sys.executable,
        str(RUN_SCRIPT),
        "--workspace-root", str(variant_work_root(variant)),
        "--stats-root", str(variant_stats_root(variant)),
        "--phase1-rounds", "0",
        "--phase2-rounds", str(PHASE2_ROUNDS_PER_VARIANT),
        "--batch-size", str(BATCH_SIZE),
        "--workers", str(WORKERS),
        "--gpus", str(GPUS),
        "--master-port", str(MASTER_PORT_BASE + selected.index(variant)),
        "--log-file", str(variant_train_log(variant)),
        "--classwise-blend", str(variant["classwise_blend"]),
        "--server-anchor", str(variant["server_anchor"]),
        "--temperature", str(variant["temperature"]),
        "--stability-lambda", str(variant["stability_lambda"]),
        "--dqa-start-phase", str(variant["dqa_start_phase"]),
    ]


def history_rows(variant: dict) -> list[dict]:
    path = variant_work_root(variant) / "history.json"
    if not path.exists():
        return []
    return json.loads(path.read_text())


def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text().strip())
    except Exception:
        return None


def pid_state(pid: int | None) -> str:
    if not pid:
        return "missing"
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return "stopped"
    except PermissionError:
        return "unknown"
    return "running"


In [5]:
# バックグラウンドで 08_3_9 を起動する。ノートブックは待たずに戻る。
if not selected:
    raise RuntimeError("No selected variants")
if not RSCOLQ_MODEL.exists():
    raise FileNotFoundError(f"Missing R-SCoLQ model: {RSCOLQ_MODEL}")

for variant in selected:
    history = history_rows(variant)
    completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
    print("\n===", variant["name"], "===")
    print(variant["description"])
    print(f"completed_phase2: {completed_phase2}/{PHASE2_ROUNDS_PER_VARIANT}")
    if completed_phase2 >= PHASE2_ROUNDS_PER_VARIANT:
        print("already completed")
        continue

    pid_path = variant_pid_path(variant)
    existing_pid = read_pid(pid_path)
    if pid_state(existing_pid) == "running":
        print("already running pid:", existing_pid)
        continue

    runner_log = variant_runner_log(variant)
    train_log = variant_train_log(variant)
    runner_log.parent.mkdir(parents=True, exist_ok=True)
    cmd = train_cmd(variant)
    env = variant_env(variant)
    print("cmd:", " ".join(cmd))
    print("runner_log:", runner_log)
    print("train_log:", train_log)

    log = runner_log.open("a", encoding="utf-8", buffering=1)
    log.write("\n" + "=" * 100 + "\n")
    log.write(" ".join(cmd) + "\n")
    log.flush()
    process = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    pid_path.write_text(str(process.pid), encoding="utf-8")
    log.close()
    print("started pid:", process.pid)
    print("status:", pid_state(process.pid))



=== a_rscolq_antidrift_budget_r003 ===
08_3_9本命。raw R-SCoLQ self-confirmationを早めに罰し、neck/head更新とpseudo loss decayで長期driftを抑える。
completed_phase2: 0/10
cmd: /opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_rscolq_anti_drift_policy.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_9_phase2_rscolq_anti_drift_policy/a_rscolq_antidrift_budget_r003 --stats-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_9_phase2_rscolq_anti_drift_policy/a_rscolq_antidrift_budget_r003 --phase1-rounds 0 --phase2-rounds 10 --batch-size 160 --workers 10 --gpus 2 --master-port 30190 --log-file /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_9_phase2_rscolq_anti_drift_policy/a_rscolq_antidrift_budget_r003_train.log --classwise-blend 0.035 --server-anchor 20.0 --temperature 3.2 --stability-lambda 0.78 --dqa-start-ph

In [6]:
# 起動直後の状態確認。
for variant in selected:
    pid = read_pid(variant_pid_path(variant))
    print(variant["name"], "pid=", pid, "state=", pid_state(pid))
    print("runner_log:", variant_runner_log(variant))
    print("train_log:", variant_train_log(variant))
    sched = variant_stats_root(variant) / "phase2_rscolq_anti_drift_policy_schedule.jsonl"
    print("schedule:", sched)


a_rscolq_antidrift_budget_r003 pid= 1972394 state= running
runner_log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_9_phase2_rscolq_anti_drift_policy/a_rscolq_antidrift_budget_r003_runner.out
train_log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_9_phase2_rscolq_anti_drift_policy/a_rscolq_antidrift_budget_r003_train.log
schedule: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_9_phase2_rscolq_anti_drift_policy/a_rscolq_antidrift_budget_r003/phase2_rscolq_anti_drift_policy_schedule.jsonl


In [7]:
# Discord通知。学習完了通知ではなく、08_3_9の起動通知。
try:
    sys.path.insert(0, str(REPO_ROOT))
    from notebook_notify import notify_discord
    notify_discord("０８＿３＿９を起動しました", title="実行開始")
except Exception as exc:
    print("notify skipped:", exc)
